<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [ ]:
%pip install -q folium pandas


In [ ]:
import folium
import pandas as pd

In [ ]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [ ]:
## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [ ]:
# Download and read the augmented SpaceX launch-site dataset.
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
spacex_df = pd.read_csv(URL)
spacex_df.head()


Now, you can take a look at what are the coordinates for each site.


In [ ]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [ ]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [ ]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [ ]:
# Create a map centered near NASA Johnson Space Center.
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Add a circle and a labeled marker for every launch site.
for _, site in launch_sites_df.iterrows():
    coordinate = [site["Lat"], site["Long"]]
    site_name = site["Launch Site"]

    folium.Circle(
        location=coordinate,
        radius=1000,
        color="#3388ff",
        fill=True,
        fill_opacity=0.25,
        popup=folium.Popup(site_name, parse_html=True)
    ).add_to(site_map)

    folium.Marker(
        location=coordinate,
        tooltip=site_name,
        icon=DivIcon(
            icon_size=(150, 30),
            icon_anchor=(0, 0),
            html=f'<div style="font-size: 12px; color: #d35400;"><b>{site_name}</b></div>'
        )
    ).add_to(site_map)

site_map


The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


**Interpretation (Task 1):** The launch sites are not all located directly on the Equator. They are in coastal regions, which is useful for launches because rockets can travel over open water and reduce risk to populated areas. The map lets you inspect the exact relationship between each site, the equator, and the coastline.


In [ ]:
# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [ ]:
spacex_df.tail(10)

Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [ ]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [ ]:
# Successful launches (class=1) are green; unsuccessful launches (class=0) are red.
spacex_df["marker_color"] = spacex_df["class"].apply(
    lambda launch_class: "green" if launch_class == 1 else "red"
)

spacex_df[["Launch Site", "class", "marker_color"]].head()


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [ ]:
# Add all launch outcome markers to the map using a marker cluster.
marker_cluster = MarkerCluster().add_to(site_map)

for _, record in spacex_df.iterrows():
    coordinate = [record["Lat"], record["Long"]]
    popup_text = (
        f"Launch Site: {record['Launch Site']}<br>"
        f"Outcome: {'Success' if record['class'] == 1 else 'Failure'}"
    )

    marker = folium.Marker(
        location=coordinate,
        popup=folium.Popup(popup_text, max_width=250),
        icon=folium.Icon(
            color="white",
            icon_color=record["marker_color"],
            icon="rocket",
            prefix="fa"
        )
    )
    marker_cluster.add_child(marker)

site_map


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


**Task 2 interpretation:** Green markers represent successful launches (`class = 1`) and red markers represent unsuccessful launches (`class = 0`). Marker clustering groups launches that share or are very close to the same coordinates. Compare the green/red distribution at each launch site to assess relative outcomes; the map visualizes outcomes but does not by itself establish why a launch succeeded or failed.


In [ ]:
# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [ ]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [ ]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [ ]:
# Example coastline point near Cape Canaveral, Florida.
# The example coordinate is supplied in the lab instructions.
coastline_lat = 28.56367
coastline_lon = -80.57163

# Use CCAFS SLC-40 as the example launch site for the distance calculation.
selected_site = launch_sites_df[
    launch_sites_df["Launch Site"].str.contains("CCAFS SLC-40", case=False, na=False)
]

if selected_site.empty:
    # Fall back to the first available launch site if the expected label differs.
    selected_site = launch_sites_df.iloc[[0]]

launch_site_lat = float(selected_site.iloc[0]["Lat"])
launch_site_lon = float(selected_site.iloc[0]["Long"])
selected_site_name = selected_site.iloc[0]["Launch Site"]

distance_coastline = calculate_distance(
    launch_site_lat, launch_site_lon, coastline_lat, coastline_lon
)

print(f"Selected launch site: {selected_site_name}")
print(f"Distance to the example coastline point: {distance_coastline:.2f} km")


In [ ]:
# Mark the selected coastline point and display its distance from the launch site.
coastline_coordinate = [coastline_lat, coastline_lon]

distance_marker = folium.Marker(
    location=coastline_coordinate,
    tooltip="Selected coastline point",
    icon=DivIcon(
        icon_size=(180, 30),
        icon_anchor=(0, 0),
        html=(
            '<div style="font-size: 12px; color: #d35400;">'
            f'<b>{distance_coastline:.2f} KM to launch site</b></div>'
        )
    )
)
site_map.add_child(distance_marker)

folium.Marker(
    location=coastline_coordinate,
    popup="Example coastline point",
    icon=folium.Icon(color="blue", icon="info-sign")
).add_to(site_map)

site_map


*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [ ]:
# Draw a line from the selected launch site to the coastline point.
launch_site_coordinate = [launch_site_lat, launch_site_lon]
coordinates = [launch_site_coordinate, coastline_coordinate]

lines = folium.PolyLine(
    locations=coordinates,
    color="red",
    weight=2,
    tooltip=f"Distance: {distance_coastline:.2f} km"
)
site_map.add_child(lines)
site_map


Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [ ]:
# Example: add a distance label for the selected coastline point.
# The same method can be repeated for a city, railway, or highway after
# recording its latitude and longitude with the map's MousePosition display.
proximity_map = folium.Map(
    location=launch_site_coordinate,
    zoom_start=13
)

folium.Marker(
    launch_site_coordinate,
    popup=selected_site_name,
    icon=folium.Icon(color="green", icon="rocket", prefix="fa")
).add_to(proximity_map)

folium.Marker(
    coastline_coordinate,
    popup="Example coastline point",
    icon=folium.Icon(color="blue", icon="info-sign")
).add_to(proximity_map)

folium.PolyLine(
    [launch_site_coordinate, coastline_coordinate],
    color="red",
    weight=3,
    tooltip=f"{distance_coastline:.2f} km"
).add_to(proximity_map)

folium.Marker(
    coastline_coordinate,
    icon=DivIcon(
        icon_size=(160, 30),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12px; color: #d35400;"><b>{distance_coastline:.2f} KM</b></div>'
    )
).add_to(proximity_map)

proximity_map


In [ ]:
# Summary of the mapped launch-site coordinates.
launch_sites_df.assign(
    Latitude=launch_sites_df["Lat"].round(4),
    Longitude=launch_sites_df["Long"].round(4)
)[["Launch Site", "Latitude", "Longitude"]]


In [ ]:
# Calculate the distance from every launch site to the same example coastline point.
# This is a straight-line (great-circle) distance, not a road or walking distance.
distance_summary = launch_sites_df.copy()
distance_summary["Distance_to_example_coastline_km"] = distance_summary.apply(
    lambda row: calculate_distance(
        row["Lat"], row["Long"], coastline_lat, coastline_lon
    ),
    axis=1
)
distance_summary[["Launch Site", "Distance_to_example_coastline_km"]].sort_values(
    "Distance_to_example_coastline_km"
).round(2)


**Task 3 interpretation:** The distance calculation uses the Haversine formula and gives straight-line distance in kilometres. The coastline point used here is the example coordinate supplied in the lab (`28.56367, -80.57163`), near Cape Canaveral, and the example calculation uses CCAFS SLC-40 when that site is present in the data. The map can be used to choose additional railway, highway, or city coordinates manually; no exact distances to those features can be stated until their coordinates are selected. In general, launch sites are commonly positioned near coastlines and away from dense urban areas for operational and safety reasons.


After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
